# 02. 형식 검증 · 음수 금액 · 날짜 현실성 · 그룹 누수 구조

발표자료가 말하는 **“발견된 품질 이슈 5가지”**가 실제로 검출되는 구간이다.

음수 금액 30%는 오류가 아니라 출금/입금 방향이고, 2003~2058년 거래일자는 시간 기반 분할이 부적절하다는 근거이며, 고객·계좌 중복 등장은 그룹 분할이 필수라는 근거다.

> ⚠️ **형식 검증은 `train` 이 아니라 `train_raw`(변환 전 문자열)에서 해야 한다.** 타입 변환 후에는 파싱 실패가 이미 `NaT`/`NaN` 으로 삼켜져 진짜 형식 위반이 안 잡힌다.

> **출처** — `원본/FDS_전처리_정리본.ipynb` (109셀 · Colab 실행본)
>
> 원본은 한 노트북에 §0~§15 를 전부 담고 있어 어디까지가 한 덩어리인지 알기 어려웠다.
> 이 저장소는 **절 경계 그대로** 5개로 나누고 **원본 실행 출력 98건을 모두 보존**했다.
> 코드는 손대지 않았다 — 셀 순서·내용 모두 원본과 동일하다.



### 재현 조건

| 항목 | 값 |
|---|---|
| 입력 | `train_final.csv` (원본 120,000행 × 64컬럼) |
| 원본 실행 경로 | `/content/train_final.csv` (Google Colab) |
| 저장소 경로 | `data/train.csv` — 용량(54MB) 때문에 미포함, `data/README.md` 참고 |
| 최종 산출물 | `X_tr/X_va/y_tr/y_va.parquet` (3차 전처리 · 58피처) + `label_encoders.pkl` · `le_target.pkl` |

> 노트북을 **순서대로(01→05)** 실행해야 한다. 앞 노트북의 `train` · `df` · `tr_idx`/`va_idx` 를
> 뒤 노트북이 이어받는 구조라, 단독 실행하면 `NameError` 가 난다.
> 각 노트북 첫 셀에 이어받는 변수를 명시해 두었다.

**변수 인계** — 01 에서 `train`, `train_raw` 를 이어받는다.

> 원본 셀 범위: `[12] ~ [23]` (총 12셀)


## 3. 형식 검증 (원본 `train_raw` 기준)

타입 변환 후의 `train` 이 아니라 변환 전 문자열을 가진 `train_raw` 에서 검증해야 진짜 형식 위반이 잡힌다.


In [ ]:
# 3-1. datetime 파싱 실패 (원본 문자열 기준)
datetime_quality = []
for col in datetime_cols:
    s = train_raw[col]
    original_missing = int(s.isna().sum())
    if pd.api.types.is_datetime64_any_dtype(s):
        parse_fail_count, note = 0, '이미 datetime (원본 문자열 없음 → 검증 생략)'
    else:
        parsed = pd.to_datetime(s, errors='coerce')
        parse_fail_count = int((s.notna() & parsed.isna()).sum())   # 값이 있었는데 파싱 후 NaN = 형식 위반
        note = '문자열 파싱 검증'
    datetime_quality.append({
        'column': col, 'original_missing': original_missing,
        'parse_fail_count': parse_fail_count,
        'parse_fail_ratio_percent': round(parse_fail_count / len(train_raw) * 100, 4),
        'note': note,
    })
display(pd.DataFrame(datetime_quality).sort_values('parse_fail_count', ascending=False))

,column,original_missing,parse_fail_count,parse_fail_ratio_percent,note
0,Customer_registration_datetime,0,0,0.0,문자열 파싱 검증
1,Account_creation_datetime,0,0,0.0,문자열 파싱 검증
2,Transaction_Datetime,0,0,0.0,문자열 파싱 검증
3,Last_atm_transaction_datetime,0,0,0.0,문자열 파싱 검증
4,Last_bank_branch_transaction_datetime,0,0,0.0,문자열 파싱 검증
5,Transaction_resumed_date,0,0,0.0,문자열 파싱 검증


In [ ]:
# 3-2. Time_difference(timedelta) 파싱 실패 (원본 문자열 기준)
s = train_raw['Time_difference']
parsed = pd.to_timedelta(s, errors='coerce')
fail = int((s.notna() & parsed.isna()).sum())
print('Time_difference 형식 위반:', fail, '건')

Time_difference 형식 위반: 0 건


## 4. 음수 금액 점검 (방향 정보 vs 오류)

In [ ]:
# 4-1. 수치형 중 음수가 존재하는 컬럼
neg_report = []
for col in numeric_cols:
    s = train[col]
    neg = int((s < 0).sum())
    if neg > 0:
        neg_report.append({
            'column': col, 'negative_count': neg,
            'negative_ratio_percent': round((s < 0).mean() * 100, 4),
            'min': s.min(), 'max': s.max(),
        })
display(pd.DataFrame(neg_report).sort_values('negative_ratio_percent', ascending=False))
# 해석: Transaction_Amount 음수는 '오류'가 아니라 입출금 '방향' 가능성 → 이후 부호 분리 + 절댓값 보존 예정

,column,negative_count,negative_ratio_percent,min,max
2,Transaction_Amount,35817,29.8475,-382480000,406690000
0,Account_initial_balance,4152,3.4600,-47002364,378294259
1,Account_balance,1661,1.3842,-45756563,408024828


In [ ]:
# 4-2. Transaction_Amount 부호 분포
amt = train['Transaction_Amount']
sign_dist = pd.Series({'음수(<0)': (amt < 0).sum(), '0': (amt == 0).sum(), '양수(>0)': (amt > 0).sum()})
display(pd.DataFrame({'count': sign_dist, 'ratio_percent': (sign_dist / len(train) * 100).round(2)}))

,count,ratio_percent
음수(<0),35817,29.85
0,2018,1.68
양수(>0),82165,68.47


## 5. 날짜 현실성 점검 (시간기반 분할 적절성 — 기획서 H4)

In [ ]:
# 5-1. datetime 컬럼별 범위 / 미래(>2025) 비중
date_report = []
for col in datetime_cols:
    s = pd.to_datetime(train[col], errors='coerce')
    future = int((s.dt.year > 2025).sum())
    date_report.append({
        'column': col, 'min_date': s.min(), 'max_date': s.max(),
        'year_min': int(s.dt.year.min()), 'year_max': int(s.dt.year.max()),
        'future_count(>2025)': future,
        'future_ratio_percent': round(future / len(train) * 100, 4),
    })
display(pd.DataFrame(date_report))

,column,min_date,max_date,year_min,year_max,future_count(>2025),future_ratio_percent
0,Customer_registration_datetime,2003-01-05 11:03:41,2024-11-16 07:31:46,2003,2024,0,0.0000
1,Account_creation_datetime,2003-01-14 02:01:26,2024-12-01 17:06:38,2003,2024,0,0.0000
2,Transaction_Datetime,2003-01-25 22:20:34,2058-06-30 19:13:38,2003,2058,39133,32.6108
3,Last_atm_transaction_datetime,2003-01-21 21:29:08,2058-06-17 12:34:39,2003,2058,37341,31.1175
4,Last_bank_branch_transaction_datetime,2003-01-22 23:38:48,2058-06-24 12:21:34,2003,2058,37415,31.1792
5,Transaction_resumed_date,2003-01-19 21:29:08,2058-05-15 01:14:42,2003,2058,38796,32.3300


In [ ]:
# 5-2. 거래일자 연도 분포
tx_year = train['Transaction_Datetime'].dt.year
display(tx_year.value_counts().sort_index().rename('count'))
# 해석: 미래(2025 초과) 연도까지 분포 → 시간 순서가 무의미 → 시간기반 분할 부적절, 층화 분할 주력 (H4)

,count
Transaction_Datetime,
2003,487
2004,813
2005,1292
2006,1688
2007,2149
2008,2670
2009,3144
2010,3448
2011,4041


## 6. 고객/계좌 반복 & 그룹 누수 구조 (분할 전략 근거 — 기획서 5단계)


In [ ]:
# 6-1. 반복 구조
n_cust = train['Customer_identification_number'].nunique()
n_acc  = train['Account_account_number'].nunique()
print('고유 고객 수:', f'{n_cust:,}')
print('고유 계좌 수:', f'{n_acc:,}')
print('고객당 평균 거래수:', round(len(train) / n_cust, 1))

cust_acc = train.groupby('Customer_identification_number')['Account_account_number'].nunique()
print('\n고객당 계좌 수 분포:')
display(cust_acc.value_counts().rename('고객수'))

고유 고객 수: 4,101
고유 계좌 수: 4,101
고객당 평균 거래수: 29.3

고객당 계좌 수 분포:


,고객수
Account_account_number,
1,4101


In [ ]:
# 6-2. 정상/사기 양쪽에 등장하는 고객 (랜덤 분할 시 누수 위험)
fraud  = train[train['Fraud_Type'] != 'm']
normal = train[train['Fraud_Type'] == 'm']
both = set(fraud['Customer_identification_number']) & set(normal['Customer_identification_number'])
print('사기 행 수:', f'{len(fraud):,}')
print('사기에 등장한 고유 고객 수:', f"{fraud['Customer_identification_number'].nunique():,}")
print('정상·사기 양쪽에 등장하는 고객 수:', f'{len(both):,}')
# 해석: 같은 고객이 train/valid 양쪽에 들어가면 누수 → '층화' vs '고객 단위 그룹 분할' 결정 필요 (다음 단계)

사기 행 수: 1,200
사기에 등장한 고유 고객 수: 1,067
정상·사기 양쪽에 등장하는 고객 수: 966
